# Example submission — random baseline (one NDIF session)

A minimal submission that wires up the **whole pipeline** end to end inside a
single remote **NDIF session**: loading the dataset, tokenizing, every model
forward pass, and your classifier all run **on NDIF as one job**, and only the
per-row `index` + `score` come back. Batching the work into one session (one
queue wait) is far more robust to NDIF's queue latency than many separate
traces.

The boilerplate lives in [`util.py`](util.py) — `run_full_session` /
`write_submission` — so this notebook is just the *method*. The baseline flips a
coin (~0.5); replace `detect` with a real signal read off the activations (see
[`../tutorials/2-predicting.ipynb`](../tutorials/2-predicting.ipynb)).

**Contract:** running this top-to-bottom must write `submission.csv` with columns
`index,deceptive,score` — one row per example:
- `index` — the row's `index` (joins your row to the held-out label).
- `deceptive` — your **binary** call, `True`/`False` (used for balanced accuracy,
  recall, false-positive rate).
- `score` — a continuous deception score in `[0, 1]` you'd threshold to get
  `deceptive` (used for AUROC).

In [ ]:
import os

# ── DO NOT CHANGE ───────────────────────────────────────────────────
# The runner sets these for each run; read them, don't hard-code or override.
#   DATASET_NAME : the eval dataset to predict on.
#   NDIF_HOST    : the NDIF cluster nnsight traces run on (set automatically;
#                  remote=True traces use it — you don't configure a host).
#   NDIF_API_KEY / HF_TOKEN : your forwarded credentials (already in the env).
DATASET_NAME = os.environ["DATASET_NAME"]

# Optional: `python submit.py --limit N` forwards N here as $ALETHEIA_LIMIT, so a
# --dry (or a real run) can score just the first N rows of each dataset for a quick
# rehearsal. Unset -> score every row. (Reading it here, not hard-coding, keeps the
# full run intact on the leaderboard, which sets no limit.)
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None

## Your method

`detect` is the one thing you write: it gets a right-padded `batch` of rows,
**opens its own `model.trace(...)` (or `model.generate(...)`)** on
`batch.input_ids` / `batch.attention_mask`, computes a `(B,)` tensor of deception
scores in `[0, 1]`, **`.save()`s it inside the trace, and returns it** (the save
is required for the value to survive). `run_full_session` does everything around
it — loading the right model for the dataset, tokenizing (via a pluggable
`preprocess`, default `util.chat_preprocess`), length-sorted batching, and
stitching the scores back into dataset order.

Inside the trace, read `model.output` for next-token logits or
`util.decoder_layers(model)[L].output` for a layer's residual stream; reduce per
row with `batch.gather_last(h)` (last real token) or `batch.pool_response(h)`
(mean over the assistant's response). Pass extra objects (a trained probe, a
layer index, …) as keyword args to `run_full_session` and they arrive in `detect`
as `**kwargs`.

In [ ]:
import sys

import torch

# util.py ships alongside the notebooks in submission/; the runner executes from
# the repo root, so add submission/ to the path before importing it.
#
# NOTE: util.py is a *convenience*, not a requirement. It bundles one reasonable
# way to run a submission (peek row 0 → build the model → one batched remote
# session → write the CSV). Depending on your method you'll likely need to
# implement some of this yourself — a custom preprocess/tokenization, a different
# batching or pooling scheme, multiple traces, generation instead of a forward
# pass — and some methods are easiest to write end to end without util.py at all.
# The only hard contract is the one in the first markdown cell: produce
# submission.csv with columns index,deceptive,score. How you get there is up to
# you; treat the helpers below as a starting point to adapt or replace.
sys.path.insert(0, "submission")
import util


def detect(model, model_id, lora_id, batch):
    """Score one batch. Open your own trace (or generate), .save() the scores
    inside it, and return them — a (B,) tensor of deception scores in [0, 1]."""
    with model.trace({"input_ids": batch.input_ids, "attention_mask": batch.attention_mask}):
        # The model's own next-token logits at each row's last real token — one
        # simple, architecture-agnostic signal. For a real probe, read activations:
        #   h = util.decoder_layers(model)[LAYER].output   # (B, seq, hidden)
        #   feats = batch.pool_response(h)                 # (B, hidden)
        logits = batch.gather_last(model.output.logits)        # (B, vocab)
        # ── your method goes here: turn `logits` (or activations) into a score ──
        scores = torch.rand(logits.shape[0]).save()           # baseline: a coin flip
    return scores


# One job, one queue wait: load + tokenize + every forward pass + your detector
# all run on NDIF; only the per-row scores come back (in dataset order).
# Keep batch_size * max_len under the cluster's ~2560-token budget — the eval
# models are ~27B, so a big batch CUDA-OOMs (8 * 256 = 2048 is safe).
scores = util.run_full_session(DATASET_NAME, detect, batch_size=8, max_len=256, limit=LIMIT)
print(f"scored {len(scores)} rows in one session")

In [ ]:
# Join scores back to each row's `index` and write index,deceptive,score.
# `deceptive` defaults to score >= 0.5 — pass deceptive=<bool array> to override.
examples = util.load_examples(DATASET_NAME)
# scores cover the first len(scores) rows (every row unless --limit/$ALETHEIA_LIMIT
# is set), in dataset order — align the index column to match.
util.write_submission(list(examples["index"])[:len(scores)], scores)